# Generate cn data and save to './data/{filename}.parquet'

In [1]:
from src.generate_data import generate_cn_data
import cProfile
import pstats

filename = "profiled_cn_run"
versions = ('v1', 'v2', 'v3', 'v4', 'v5', 'v6')
s_steps = [100, 500] + list(range(1000, 5000, 1000))
s_steps = tuple(s_steps)
t_steps = [100, 500] + list(range(1000, 5000, 1000))
t_steps = tuple(t_steps)
n_reps = 5  # for the entire run
warmup_runs = 3 # per method

K = 100
S0 = 100
T = 3
r = 0.02
sigma = 0.4

# grid uses x = logS
print(f"executing {len(s_steps) * len(t_steps) * len(versions) * n_reps} runs in total")
print(f"for x grid steps in: {s_steps}")
print(f"for t grid steps in: {t_steps}")
print(f"for versions in: {versions}")
print(f"repeating for {n_reps} repetitions")


executing 1080 runs in total
for x grid steps in: (100, 500, 1000, 2000, 3000, 4000)
for t grid steps in: (100, 500, 1000, 2000, 3000, 4000)
for versions in: ('v1', 'v2', 'v3', 'v4', 'v5', 'v6')
repeating for 5 repetitions


In [2]:
print("performing warmup cn runs..")
from src.options import EuropeanOption
V = EuropeanOption(100, 3, type='call')
for i in range(warmup_runs):
    for v in versions:
        V.price_CN(S0=S0, r=r, sigma=sigma, s_steps=500, t_steps=500, version=v)


performing warmup fdm runs..


In [3]:
cn_config = dict(
    K = K,
    T = T,
    S0 = S0,
    r = r,
    sigma = sigma,
    n_reps = n_reps,
    s_steps = s_steps,
    t_steps = t_steps,
    filename = filename,
    versions = versions
)



# generate data
profiler = cProfile.Profile()
profiler.enable()
df_cn = generate_cn_data(**cn_config)
profiler.disable()

stats = pstats.Stats(profiler)
stats.strip_dirs()
stats.sort_stats("cumulative")
profiler.dump_stats(f'./data/{filename}.prof')

versions chosen: ('v1', 'v2', 'v3', 'v4', 'v5', 'v6')


Running:   0%|          | 0/1080 [00:00<?, ?run/s]